# Adaptive Biome Scheduler V3

## Distributed endgame and cash-flow crop rotation

This is an independently written, observation-driven Kaggriculture agent. It does not replay a recorded episode or identify a particular opponent. Every action is recomputed from the current farm, private inventory, public opponent farm, town demand, market prices, crop ages, animal condition, remaining day, and worker positions.

### V3 improvements

- **Distributed terminal zones:** final-day cells are divided among workers so they do not all chase the same harvest.
- **Batched return trips:** workers collect a compact batch before returning to the shed instead of carrying one item at a time.
- **Earlier expansion rotation:** empty expansion cells rotate from long-horizon strawberries to fast wheat while existing strawberries continue producing.
- **Live portfolio response:** the animal cohort and sales remain responsive to public supply pressure and current prices.
- **No fixed replay:** there are no episode IDs, submission IDs, usernames, opponent-specific branches, or per-turn action arrays.

## Authorship, provenance, and licensing

The code in this notebook was independently developed from:

1. Kaggle's official Kaggriculture mechanics and action schema;
2. diagnostics from the author's own agent replays;
3. original simulation, ablation, and head-to-head testing.

No public competitor notebook code was copied or edited to create this implementation. AI-assisted debugging and test generation were used, while the author reviewed the final source and remains responsible for the submission.

- **Source code:** Apache License 2.0
- **Notebook narrative:** CC BY 4.0

Keep this section when publishing the notebook.

In [ ]:
from pathlib import Path
import ast
import importlib
import re
import tarfile

print("Adaptive Biome Scheduler V3 workspace ready")

## Write the standalone agent

Kaggle imports the `agent(obs, configuration=None)` entry point from `main.py`.

In [ ]:
%%writefile main.py
# Copyright 2026 Ömer Kiraz
# SPDX-License-Identifier: Apache-2.0
"""Adaptive Biome Scheduler V3 — distributed endgame and crop rotation policy.

This policy is independently written from the public Kaggriculture mechanics.
It contains no replay actions, episode identifiers, opponent names, or copied
competition policy. Every decision is reconstructed from the live observation.
"""
from math import ceil

CROPS = {
    "WHEAT": {"seed": 10, "first": 2, "last": 4, "ongoing": False},
    "CARROT": {"seed": 20, "first": 2, "last": 3, "ongoing": False},
    "TOMATO": {"seed": 50, "first": 8, "last": 11, "ongoing": True},
    "STRAWBERRY": {"seed": 100, "first": 10, "last": 16, "ongoing": True},
    "MELON": {"seed": 80, "first": 10, "last": 12, "ongoing": False},
}
ANIMAL_COST = {"COW": 400, "SHEEP": 500}
ANIMAL_PRODUCT = {"COW": "MILK", "SHEEP": "WOOL"}
BASE_PRICE = {
    "WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120,
    "MELON": 250, "EGG": 50, "MILK": 160, "WOOL": 200,
    "FERTILIZER": 100,
}
PRODUCTS = tuple(BASE_PRICE)
LAND_COSTS = (1000, 2000, 4000)
SHED_TILES = ((4, 4), (5, 4), (4, 5), (5, 5))

# Compact, symmetric ranch layout. The order is also the capital deployment
# order: four animals at launch, two more on day 4, four after NE expansion,
# and four after SW expansion.
ANIMAL_SLOTS = (
    ((4, 4), "COW"), ((4, 3), "COW"), ((3, 4), "COW"), ((3, 3), "SHEEP"),
    ((2, 4), "COW"), ((4, 2), "SHEEP"),
    ((5, 4), "COW"), ((5, 3), "SHEEP"), ((6, 4), "COW"), ((6, 3), "SHEEP"),
    ((4, 5), "SHEEP"), ((3, 5), "SHEEP"), ((4, 6), "SHEEP"), ((3, 6), "SHEEP"),
)
ANIMAL_COORDS = {coord for coord, _ in ANIMAL_SLOTS}
STRAWBERRY_EVENTS = (10, 12, 14, 16)

SHOP_PRODUCTS = {
    "BAKERY": ("EGG", "WHEAT"),
    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),
    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),
    "YARN_STORE": ("WOOL",),
    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),
    "PET_CAFE": ("CARROT",),
    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),
    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),
}


def _get(obj, key, default=None):
    return obj.get(key, default) if isinstance(obj, dict) else getattr(obj, key, default)


def _dict(value):
    return value if isinstance(value, dict) else {}


def _distance(a, b):
    return abs(int(a[0]) - int(b[0])) + abs(int(a[1]) - int(b[1]))


def _move(position, target):
    x, y = int(position[0]), int(position[1])
    tx, ty = int(target[0]), int(target[1])
    # Take the longer axis first; this shortens most trips and naturally varies
    # routes between workers assigned to different blocks.
    if abs(tx - x) >= abs(ty - y):
        if x < tx: return ["EAST"]
        if x > tx: return ["WEST"]
        if y < ty: return ["SOUTH"]
        if y > ty: return ["NORTH"]
    else:
        if y < ty: return ["SOUTH"]
        if y > ty: return ["NORTH"]
        if x < tx: return ["EAST"]
        if x > tx: return ["WEST"]
    return ["PASS"]


def _nearest_shed(position, farm=None):
    candidates = list(SHED_TILES)
    if farm is not None:
        tiles = farm.get("tiles", [])
        open_tiles = [p for p in candidates if p[1] < len(tiles) and p[0] < len(tiles[p[1]]) and tiles[p[1]][p[0]] != "LOCKED"]
        if open_tiles:
            candidates = open_tiles
    return min(candidates, key=lambda p: (_distance(position, p), p[1], p[0]))


def _quadrant(x, y):
    return ("N" if y < 5 else "S") + ("W" if x < 5 else "E")


def _slot_count(day):
    if day < 4: return 4
    if day < 7: return 6
    if day < 10: return 10
    return 14


FLEX_ANIMAL_COORDS = ((5, 4), (6, 4))


def _flex_animal_choice(obs, farm):
    """Choose one stable expansion cohort from public supply pressure."""
    tiles = farm.get("tiles", [])
    # Once either flexible slot is occupied, preserve that cohort forever.
    for x, y in FLEX_ANIMAL_COORDS:
        if y < len(tiles) and x < len(tiles[y]):
            tile = tiles[y][x]
            if isinstance(tile, dict) and tile.get("animal") in ("COW", "SHEEP"):
                return tile["animal"]

    farms = _get(obs, "farms", []) or []
    pid = int(_get(obs, "player", 0) or 0)
    opponent_cows = opponent_sheep = 0
    if len(farms) == 2:
        opponent = farms[1 - pid]
        for row in opponent.get("tiles", []):
            for tile in row:
                if isinstance(tile, dict):
                    if tile.get("animal") == "COW": opponent_cows += 1
                    elif tile.get("animal") == "SHEEP": opponent_sheep += 1

    prices = _dict(_dict(_get(obs, "market", {})).get("prices", {}))
    milk_ratio = float(prices.get("MILK", BASE_PRICE["MILK"]) or BASE_PRICE["MILK"]) / BASE_PRICE["MILK"]
    wool_ratio = float(prices.get("WOOL", BASE_PRICE["WOOL"]) or BASE_PRICE["WOOL"]) / BASE_PRICE["WOOL"]

    # A rival with a clear cow surplus is likely to pressure milk. Otherwise
    # choose sheep only when the live normalized wool price is materially
    # stronger. This is a market response, not an opponent identity rule.
    if opponent_cows - opponent_sheep >= 3 or wool_ratio > 1.18 * milk_ratio:
        return "SHEEP"
    return "COW"


def _active_slots(obs, farm, day):
    result = []
    flexible = _flex_animal_choice(obs, farm)
    for coord, animal in ANIMAL_SLOTS[:_slot_count(day)]:
        x, y = coord
        tiles = farm.get("tiles", [])
        if y < len(tiles) and x < len(tiles[y]) and tiles[y][x] != "LOCKED":
            wanted = flexible if coord in FLEX_ANIMAL_COORDS else animal
            # Existing livestock is always retained, even if prices later move.
            tile = tiles[y][x]
            if isinstance(tile, dict) and tile.get("animal") in ("COW", "SHEEP"):
                wanted = tile["animal"]
            result.append((coord, wanted))
    return result


def _crop_layout():
    """Return independent coordinate ranks for three productive quadrants."""
    by_q = {"NW": [], "NE": [], "SW": []}
    for y in range(10):
        for x in range(10):
            q = _quadrant(x, y)
            if q not in by_q or (x, y) in ANIMAL_COORDS:
                continue
            by_q[q].append((x, y))
    for q in by_q:
        by_q[q].sort(key=lambda p: (_distance(p, (4, 4)), p[1], p[0]))
    return by_q


CROP_BLOCKS = _crop_layout()
CROP_LAYOUT = {
    coord: quadrant
    for quadrant, coords in CROP_BLOCKS.items()
    for coord in coords
}
CROP_RANK = {
    coord: rank
    for _quadrant_name, coords in CROP_BLOCKS.items()
    for rank, coord in enumerate(coords)
}


def _intended_crop(coord, day):
    """Stage land use from cash flow to durable premium production."""
    coord = tuple(coord)
    quadrant = CROP_LAYOUT.get(coord)
    rank = CROP_RANK.get(coord, 0)
    if quadrant is None:
        return None

    if quadrant == "NW":
        # Launch with eleven fast wheat loops and eight high-value melons.
        if day <= 3:
            return "WHEAT" if rank < 11 else "MELON"
        # Recycle mature wheat into a balanced bridge portfolio.
        if day <= 7:
            if rank < 6: return "WHEAT"
            if rank < 16: return "MELON"
            return "STRAWBERRY"
        # Long middle game: a small feed block, ten melons and strawberries.
        if day <= 17:
            if rank < 4: return "WHEAT"
            if rank < 14: return "MELON"
            return "STRAWBERRY"
        # Existing melons are maintained to harvest, but empty melon cells roll
        # into wheat. Strawberries keep producing until the late conversion.
        if rank < 14:
            return "WHEAT" if day <= 27 else None
        return "STRAWBERRY" if day <= 24 else ("WHEAT" if day <= 27 else None)

    # Expansion quadrants compound as strawberries through the profitable
    # middle game, then switch to quick wheat for the final harvest window.
    if day <= 12:
        return "STRAWBERRY"
    return "WHEAT" if day <= 27 else None


def _inventory_list(private, count):
    values = private.get("inventories", []) or []
    return [values[i] if i < len(values) and isinstance(values[i], dict) else {} for i in range(count)]


def _total_item(private, item):
    total = int(private.get("shed", {}).get(item, 0) or 0)
    for inv in private.get("inventories", []) or []:
        if isinstance(inv, dict):
            total += int(inv.get(item, 0) or 0)
    return total


def _farm_summary(farm):
    plants = {c: 0 for c in CROPS}
    animals = {"COW": 0, "SHEEP": 0}
    ready = {p: 0 for p in PRODUCTS}
    weeds = 0
    for row in farm.get("tiles", []):
        for tile in row:
            if not isinstance(tile, dict):
                continue
            if tile.get("kind") == "PLANT":
                crop = tile.get("crop")
                if crop in plants:
                    plants[crop] += 1
                    ready[crop] += max(0, int(tile.get("yield_units", 0) or 0))
            elif "animal" in tile:
                animal = tile.get("animal")
                if animal in animals:
                    animals[animal] += 1
                    ready[ANIMAL_PRODUCT[animal]] += max(0, int(tile.get("yield_units", 0) or 0))
            elif tile.get("kind") == "WEED":
                weeds += 1
    return plants, animals, ready, weeds


def _shop_rate(obs, product):
    town = _dict(_get(obs, "town", {}))
    rate = 0
    for shop in town.get("unlocked_shops", []) or []:
        products = SHOP_PRODUCTS.get(shop, ())
        if product in products:
            rate += 2 if len(products) == 1 else 1
    return rate


def _desired_actors(day):
    if day < 4: return 5
    if day < 7: return 7
    if day < 10: return 10
    return 13


def _ranch_worker_count(active_slot_count, actor_count):
    # Early capital is won by getting the first crop block online quickly.
    # Two ranch workers can safely establish and service six animals; labor
    # expands only as the portfolio grows.
    desired = 2 if active_slot_count <= 6 else (3 if active_slot_count <= 10 else 5)
    return max(1, min(desired, max(1, actor_count - 1)))


def _animal_owned(private, farm, animal):
    count = _total_item(private, animal)
    for row in farm.get("tiles", []):
        for tile in row:
            if isinstance(tile, dict) and tile.get("animal") == animal:
                count += 1
    return count


def _inventory_total(inv):
    return sum(max(0, int(v or 0)) for v in inv.values())


def _drop_action(farm, position, inv, hour, threshold=8):
    total = _inventory_total(inv)
    if total <= 0:
        return None
    target = _nearest_shed(position, farm)
    if tuple(position) == tuple(target):
        if total >= threshold or hour >= 20:
            return ["DROP"]
        return None
    if total >= threshold or hour >= 20:
        return _move(position, target)
    return None


def _ranch_action(obs, farm, private, position, inv, assigned_slots):
    day = int(_get(obs, "day", 0) or 0)
    hour = int(_get(obs, "hour", 0) or 0)
    tiles = farm.get("tiles", [])
    shed = private.get("shed", {})

    placed = []
    for coord, animal in assigned_slots:
        x, y = coord; tile = tiles[y][x]
        if isinstance(tile, dict) and tile.get("animal") == animal:
            placed.append((coord, animal, tile))

    # Existing livestock always outranks expansion. Losing an animal destroys
    # both its capital and every future fertilizer/product stream.
    unfed = [(c, a, t) for c, a, t in placed if not bool(t.get("fed_today", False))]
    if unfed and int(inv.get("WHEAT", 0) or 0) <= 0:
        target = _nearest_shed(position, farm)
        available = int(shed.get("WHEAT", 0) or 0)
        if tuple(position) == tuple(target) and available > 0:
            return ["PICKUP", "WHEAT", min(max(2, len(unfed) + 1), available)]
        return _move(position, target)

    urgent = []
    for coord, animal, tile in placed:
        dist = _distance(position, coord)
        if not bool(tile.get("fed_today", False)) and int(inv.get("WHEAT", 0) or 0) > 0:
            urgent.append((250 - 4 * dist, -dist, coord, ["FEED"]))
        units = max(0, int(tile.get("yield_units", 0) or 0))
        if units >= 3:
            urgent.append((244 - 4 * dist, -dist, coord, ["HARVEST"]))
        if not bool(tile.get("cared_today", False)):
            urgent.append((228 - 4 * dist, -dist, coord, ["CARE"]))
    if urgent:
        _, _, target, action = max(urgent)
        return action if tuple(position) == tuple(target) else _move(position, target)

    # Once all live animals are safe, finish placement of purchased stock.
    for animal in ("COW", "SHEEP"):
        if int(inv.get(animal, 0) or 0) <= 0:
            continue
        candidates = []
        for coord, wanted in assigned_slots:
            if wanted != animal: continue
            x, y = coord; tile = tiles[y][x]
            if isinstance(tile, dict) and tile.get("kind") == "PASTURE" and "animal" not in tile:
                candidates.append(coord)
        if candidates:
            target = min(candidates, key=lambda p: (_distance(position, p), p[1], p[0]))
            return ["PLACE", animal] if tuple(position) == tuple(target) else _move(position, target)

    missing_build = []
    for coord, _animal in assigned_slots:
        x, y = coord
        if tiles[y][x] is None:
            missing_build.append(coord)
    if missing_build:
        target = min(missing_build, key=lambda p: (_distance(position, p), p[1], p[0]))
        return ["BUILD_PASTURE"] if tuple(position) == tuple(target) else _move(position, target)

    for animal in ("COW", "SHEEP"):
        gaps = 0
        for coord, wanted in assigned_slots:
            if wanted != animal: continue
            x, y = coord; tile = tiles[y][x]
            if isinstance(tile, dict) and tile.get("kind") == "PASTURE" and "animal" not in tile:
                gaps += 1
        if gaps > 0 and int(shed.get(animal, 0) or 0) > 0:
            target = _nearest_shed(position, farm)
            if tuple(position) == tuple(target):
                return ["PICKUP", animal, min(gaps, int(shed.get(animal, 0)))]
            return _move(position, target)

    secondary = []
    for coord, animal, tile in placed:
        dist = _distance(position, coord)
        units = max(0, int(tile.get("yield_units", 0) or 0))
        if units > 0:
            secondary.append((220 - 4 * dist, -dist, coord, ["HARVEST"]))
        if bool(tile.get("fertilizer_available", False)):
            secondary.append((210 - 4 * dist, -dist, coord, ["COLLECT_FERTILIZER"]))
    if secondary:
        _, _, target, action = max(secondary)
        return action if tuple(position) == tuple(target) else _move(position, target)

    drop = _drop_action(farm, position, inv, hour, threshold=6)
    return drop or ["PASS"]

def _strawberry_fertilizer_needed(tile, day):
    planted = int(tile.get("planted_day", day))
    age = day - planted
    active_until = int(tile.get("fertilized_until_day", -1))
    # Two well-timed applications can cover all four production events. A
    # one-day recovery window prevents a missed route from losing the rest of
    # an event pair.
    if age in (10, 11):
        return active_until < planted + 12
    if age in (14, 15):
        return active_until < planted + 16
    return False


def _crop_task(tile, coord, intended, day, hour, inv, position):
    if isinstance(tile, dict) and tile.get("kind") == "WEED":
        return (220, coord, ["DIG"])
    if tile is None:
        if intended is not None and hour <= 20:
            return (70, coord, ["PLANT", intended])
        return None
    if not (isinstance(tile, dict) and tile.get("kind") == "PLANT"):
        return None

    crop = tile.get("crop")
    if crop not in CROPS:
        return None
    planted_day = tile.get("planted_day", day)
    age = day - int(day if planted_day is None else planted_day)
    units = max(0, int(tile.get("yield_units", 0) or 0))
    watered = bool(tile.get("watered_today", False))
    dry = max(0, int(tile.get("consecutive_unwatered", 0) or 0))

    if crop == "STRAWBERRY":
        if units >= 3:
            return (205, coord, ["HARVEST"])
        if _strawberry_fertilizer_needed(tile, day) and int(inv.get("FERTILIZER", 0) or 0) > 0:
            return (198, coord, ["FERTILIZE"])
        production_today = age in STRAWBERRY_EVENTS
        if not watered and (age == 0 or dry >= 1 or production_today or hour >= 18):
            return (196 if production_today or dry >= 1 else 150, coord, ["WATER"])
        if units > 0 and (age >= 12 or day >= 28):
            return (175, coord, ["HARVEST"])
        # Remove exhausted plants once their last held output is collected.
        if age >= 18 and units <= 0 and intended != "STRAWBERRY":
            return (120, coord, ["DIG"])
        return None

    if crop == "MELON":
        # A fully watered melon reaches its six-unit cap on age 10. Harvesting
        # at the cap unlocks cash two days earlier without sacrificing yield.
        if age >= 10 and units >= 6:
            if not watered:
                return (202, coord, ["WATER"])
            return (207, coord, ["HARVEST"])
        if age >= 12:
            if not watered:
                return (202, coord, ["WATER"])
            if units > 0:
                return (207, coord, ["HARVEST"])
        if day >= 29 and age >= 10 and units > 0:
            return (204, coord, ["HARVEST"])
        bonus = age >= 6
        if not watered and (age == 0 or dry >= 1 or bonus or hour >= 19):
            return (194 if bonus or dry >= 1 else 145, coord, ["WATER"])
        return None

    if crop == "WHEAT":
        harvest_age = 3 if day <= 6 else 4
        if age >= harvest_age:
            if not watered:
                return (202, coord, ["WATER"])
            if units > 0:
                return (207, coord, ["HARVEST"])
        if day >= 29 and age >= 2 and units > 0:
            return (204, coord, ["HARVEST"])
        bonus = age >= 2
        if not watered and (age == 0 or dry >= 1 or bonus or hour >= 19):
            return (194 if bonus or dry >= 1 else 145, coord, ["WATER"])
        return None

    # Any unexpected crop is safely maintained and harvested rather than dug.
    if not watered and (age == 0 or dry >= 1 or hour >= 19):
        return (170, coord, ["WATER"])
    if units > 0 and age >= CROPS[crop]["last"]:
        return (165, coord, ["HARVEST"])
    return None


def _assigned_crop_coords(farm, worker_index, worker_count):
    available = []
    tiles = farm.get("tiles", [])
    for coord in CROP_LAYOUT:
        x, y = coord
        if y < len(tiles) and x < len(tiles[y]) and tiles[y][x] != "LOCKED":
            available.append(coord)
    # Serpentine ordering gives each worker one compact contiguous block.
    available.sort(key=lambda p: (p[1], p[0] if p[1] % 2 == 0 else -p[0]))
    if worker_count <= 1:
        return available
    size = int(ceil(len(available) / float(worker_count)))
    start = worker_index * size
    return available[start:start + size]


def _crop_action(obs, farm, private, position, inv, coords, seed_budget):
    day = int(_get(obs, "day", 0) or 0)
    hour = int(_get(obs, "hour", 0) or 0)
    tiles = farm.get("tiles", [])

    # Before production days, take a small fertilizer packet from the shed.
    fertilizer_jobs = 0
    for x, y in coords:
        tile = tiles[y][x]
        if isinstance(tile, dict) and tile.get("crop") == "STRAWBERRY" and _strawberry_fertilizer_needed(tile, day):
            fertilizer_jobs += 1
    if fertilizer_jobs > 0 and int(inv.get("FERTILIZER", 0) or 0) <= 0:
        target = _nearest_shed(position, farm)
        available = int(private.get("shed", {}).get("FERTILIZER", 0) or 0)
        if tuple(position) == tuple(target) and available > 0:
            return ["PICKUP", "FERTILIZER", min(fertilizer_jobs, 4, available)]
        if available > 0:
            return _move(position, target)

    candidates = []
    for coord in coords:
        x, y = coord
        intended = _intended_crop(coord, day)
        task = _crop_task(tiles[y][x], coord, intended, day, hour, inv, position)
        if task is None:
            continue
        priority, target, action = task
        if action[0] == "PLANT" and int(seed_budget.get(action[1], 0) or 0) <= 0:
            continue
        distance = _distance(position, target)
        candidates.append((priority - 3 * distance, priority, -distance, -target[1], -target[0], target, action))
    if candidates:
        _, _, _, _, _, target, action = max(candidates)
        if tuple(position) == tuple(target):
            if action[0] == "PLANT":
                seed_budget[action[1]] = max(0, int(seed_budget.get(action[1], 0)) - 1)
            return action
        return _move(position, target)

    drop = _drop_action(farm, position, inv, hour, threshold=9)
    return drop or ["PASS"]


def _terminal_action(obs, farm, position, inv, worker_index=0, worker_count=1):
    hour = int(_get(obs, "hour", 0) or 0)
    tiles = farm.get("tiles", [])
    unlocked_cells = []
    for yy, row in enumerate(tiles):
        xs = range(len(row)) if yy % 2 == 0 else range(len(row) - 1, -1, -1)
        for xx in xs:
            if row[xx] != "LOCKED":
                unlocked_cells.append((xx, yy))
    owner_map = {}
    count_cells = max(1, len(unlocked_cells))
    for rank, cell in enumerate(unlocked_cells):
        owner_map[cell] = min(max(0, worker_count - 1), int(rank * max(1, worker_count) / count_cells))
    total = _inventory_total(inv)
    shed = _nearest_shed(position, farm)
    distance_home = _distance(position, shed)
    # Carry a compact batch before returning. Immediate one-item returns waste
    # most of the final day on travel, especially in expansion quadrants.
    must_bank = total > 0 and (total >= 12 or hour + distance_home + 1 >= 23)
    if must_bank:
        if tuple(position) == tuple(shed): return ["DROP"]
        return _move(position, shed)

    candidates = []
    for y, row in enumerate(tiles):
        for x, tile in enumerate(row):
            if worker_count > 1 and owner_map.get((x, y), 0) != worker_index:
                continue
            if not isinstance(tile, dict): continue
            if tile.get("kind") == "PLANT":
                crop = tile.get("crop")
                units = max(0, int(tile.get("yield_units", 0) or 0))
                planted_day = tile.get("planted_day", 29)
                age = 29 - int(29 if planted_day is None else planted_day)
                if crop in CROPS and units > 0 and age >= CROPS[crop]["first"]:
                    home = _distance((x, y), _nearest_shed((x, y), farm))
                    dist = _distance(position, (x, y))
                    if hour + dist + home + 2 <= 23:
                        value = units * BASE_PRICE.get(crop, 1)
                        candidates.append((value / (dist + 1.0), -dist, (x, y), ["HARVEST"]))
            elif "animal" in tile and int(tile.get("yield_units", 0) or 0) > 0:
                home = _distance((x, y), _nearest_shed((x, y), farm))
                dist = _distance(position, (x, y))
                if hour + dist + home + 2 <= 23:
                    item = ANIMAL_PRODUCT.get(tile.get("animal"), "MILK")
                    value = int(tile.get("yield_units", 0)) * BASE_PRICE[item]
                    candidates.append((value / (dist + 1.0), -dist, (x, y), ["HARVEST"]))
            if isinstance(tile, dict) and "animal" in tile and tile.get("fertilizer_available", False):
                home = _distance((x, y), _nearest_shed((x, y), farm))
                dist = _distance(position, (x, y))
                if hour + dist + home + 2 <= 23:
                    candidates.append((100.0 / (dist + 1.0), -dist, (x, y), ["COLLECT_FERTILIZER"]))
    if candidates:
        _, _, target, action = max(candidates)
        return action if tuple(position) == tuple(target) else _move(position, target)
    if total > 0:
        return ["DROP"] if tuple(position) == tuple(shed) else _move(position, shed)
    return ["PASS"]


def _unit_actions(obs, farm, private):
    positions = [farm.get("farmer", [4, 4])] + list(farm.get("hands", []) or [])
    inventories = _inventory_list(private, len(positions))
    day = int(_get(obs, "day", 0) or 0)
    seed_budget = dict(private.get("seeds", {}) or {})

    if day >= 29:
        actions = [_terminal_action(obs, farm, pos, inv, idx, len(positions)) for idx, (pos, inv) in enumerate(zip(positions, inventories))]
        return actions[0], actions[1:]

    slots = _active_slots(obs, farm, day)
    ranch_n = _ranch_worker_count(len(slots), len(positions))
    crop_n = max(0, len(positions) - ranch_n)
    actions = []
    for idx, (position, inv) in enumerate(zip(positions, inventories)):
        if idx < ranch_n:
            chunk = int(ceil(len(slots) / float(max(1, ranch_n))))
            assigned = slots[idx * chunk:(idx + 1) * chunk]
            action = _ranch_action(obs, farm, private, position, inv, assigned)
        else:
            worker_index = idx - ranch_n
            coords = _assigned_crop_coords(farm, worker_index, max(1, crop_n))
            action = _crop_action(obs, farm, private, position, inv, coords, seed_budget)
        actions.append(action)
    return actions[0], actions[1:]


def _fib(index):
    a, b = 1, 1
    for _ in range(index):
        a, b = b, a + b
    return a


def _hire_cost(start_index, count):
    return sum(_fib(start_index + i) for i in range(max(0, count)))


def _planned_seed_need(farm, private, day):
    need = {c: 0 for c in CROPS}
    tiles = farm.get("tiles", [])
    for coord in CROP_LAYOUT:
        x, y = coord
        if y >= len(tiles) or x >= len(tiles[y]) or tiles[y][x] == "LOCKED":
            continue
        intended = _intended_crop(coord, day)
        tile = tiles[y][x]
        if intended and (tile is None or (isinstance(tile, dict) and tile.get("kind") == "WEED")):
            need[intended] += 1
    for crop in need:
        need[crop] = max(0, need[crop] - int(private.get("seeds", {}).get(crop, 0) or 0))
    return need


def _sell_orders(obs, farm, private, limit):
    if limit <= 0: return []
    day = int(_get(obs, "day", 0) or 0)
    hour = int(_get(obs, "hour", 0) or 0)
    shed = private.get("shed", {})
    prices = _dict(_dict(_get(obs, "market", {})).get("prices", {}))
    plants, animals, _ready, _weeds = _farm_summary(farm)
    animal_n = sum(animals.values())
    strawberry_n = plants.get("STRAWBERRY", 0)
    total_shed = sum(max(0, int(v or 0)) for v in shed.values())
    pressure = total_shed >= 78

    # Public-farm similarity is a generic race signal, not an identity check.
    farms = _get(obs, "farms", []) or []
    pid = int(_get(obs, "player", 0) or 0)
    mirror_like = False
    if len(farms) == 2:
        op, oa, _or, _ow = _farm_summary(farms[1 - pid])
        distance = sum(abs(plants[c] - op[c]) for c in plants) + 2 * sum(abs(animals[a] - oa[a]) for a in animals)
        mirror_like = distance <= 4 and day >= 8

    sale_window = hour % 4 == (0 if mirror_like else 1)
    if not (sale_window or pressure or day >= 28):
        return []

    candidates = []
    for item in PRODUCTS:
        amount = max(0, int(shed.get(item, 0) or 0))
        if amount <= 0: continue
        price = float(prices.get(item, BASE_PRICE[item]) or BASE_PRICE[item])
        ratio = price / float(BASE_PRICE[item])
        reserve = 0
        if item == "WHEAT": reserve = max(animal_n, len(_active_slots(obs, farm, day))) * 2 + 8
        if item == "FERTILIZER":
            imminent = 0
            for row in farm.get("tiles", []):
                for tile in row:
                    if isinstance(tile, dict) and tile.get("crop") == "STRAWBERRY" and _strawberry_fertilizer_needed(tile, day):
                        imminent += 1
            reserve = min(32, max(4, imminent + 4))
        sellable = max(0, amount - reserve)
        if day >= 29: sellable = amount
        if sellable <= 0: continue

        if item == "FERTILIZER":
            quantity = sellable  # no town demand: earlier sales dominate waiting
            urgency = 18 + ratio
        elif day >= 28 or pressure:
            quantity = sellable
            urgency = 16 + ratio
        else:
            demand = _shop_rate(obs, item)
            quantity = min(sellable, max(4, 4 + 2 * demand))
            if item in ("MELON", "WOOL", "STRAWBERRY", "MILK") and ratio < 0.45:
                quantity = min(quantity, 3)
            urgency = 5 * ratio + demand + (2 if item in ("MILK", "WOOL", "STRAWBERRY", "MELON") else 0)
        candidates.append((urgency, price * quantity, item, quantity))
    candidates.sort(reverse=True)
    return [["SELL", item, quantity] for _, _, item, quantity in candidates[:limit]]


def _market_orders(obs, farm, private):
    day = int(_get(obs, "day", 0) or 0)
    hour = int(_get(obs, "hour", 0) or 0)
    money = float(farm.get("money", 0.0) or 0.0)
    orders = []

    # Spread daily hiring over the first three turns so market operations remain
    # available under the ten-order cap.
    desired = _desired_actors(day)
    current = 1 + len(farm.get("hands", []) or [])
    gap = max(0, desired - current)
    hire_now = min(gap, 5 if hour <= 1 else (3 if hour == 2 else 0))
    if hire_now > 0:
        start = int(farm.get("hires_today", 0) or 0)
        affordable = 0
        running = 0
        for i in range(hire_now):
            cost = _fib(start + i)
            if running + cost > money: break
            running += cost; affordable += 1
        orders.extend([["HIRE"] for _ in range(affordable)])
        money -= running

    # Sales precede purchases so fresh revenue can fund the same turn's inputs.
    sales = _sell_orders(obs, farm, private, 10 - len(orders))
    orders.extend(sales)
    # Conservative revenue estimate; actual lockstep prices can only make later
    # purchases safer, and failed purchases are harmless no-ops.
    prices = _dict(_dict(_get(obs, "market", {})).get("prices", {}))
    for order in sales:
        money += float(prices.get(order[1], BASE_PRICE[order[1]])) * int(order[2])

    # Buy only the first three quadrants. The fourth costs more than its short
    # horizon can reliably repay in this production system.
    unlocked = len(farm.get("unlocked_quadrants", ["NW"]))
    land_index = unlocked - 1
    land_due = (land_index == 0 and day >= 7) or (land_index == 1 and day >= 10)
    if land_due and land_index < 2 and len(orders) < 10:
        cost = LAND_COSTS[land_index]
        if money >= cost + 900:
            orders.append(["BUY_LAND"]); money -= cost

    # Stage the 8-cow / 6-sheep portfolio from live counts and inventories.
    active = _active_slots(obs, farm, day)
    desired_by_type = {"COW": 0, "SHEEP": 0}
    for _coord, animal in active: desired_by_type[animal] += 1
    for animal in ("COW", "SHEEP"):
        gap = max(0, desired_by_type[animal] - _animal_owned(private, farm, animal))
        if gap > 0 and len(orders) < 10:
            affordable = min(gap, int(max(0.0, money - 350.0) // ANIMAL_COST[animal]))
            if affordable > 0:
                orders.append(["BUY_ANIMAL", animal, affordable])
                money -= affordable * ANIMAL_COST[animal]

    # Wheat is both feed insurance and a liquid staple. Keep roughly two days
    # of feed across shed and carried inventories.
    _plants, animals, _ready, _weeds = _farm_summary(farm)
    animal_n = sum(animals.values()) + sum(max(0, desired_by_type[a] - animals[a]) for a in animals)
    wheat_have = _total_item(private, "WHEAT")
    feed_target = max(10, animal_n * 2 + 6)
    if wheat_have < feed_target and len(orders) < 10 and money > 250:
        quantity = min(30, feed_target - wheat_have)
        price = float(prices.get("WHEAT", 25) or 25)
        affordable = min(quantity, int(max(0.0, money - 180.0) // max(1.0, price)))
        if affordable > 0:
            orders.append(["BUY_PRODUCT", "WHEAT", affordable])
            money -= affordable * price

    # Buy seeds by current empty planned cells, not by a fixed turn trace.
    need = _planned_seed_need(farm, private, day)
    # Priority reflects cash-flow and long-horizon value.
    priority = ("WHEAT", "MELON", "STRAWBERRY") if day < 5 else ("STRAWBERRY", "MELON", "WHEAT")
    for crop in priority:
        if len(orders) >= 10: break
        quantity = need.get(crop, 0)
        if quantity <= 0: continue
        # Buy in rolling batches to preserve operating capital.
        cap = 12 if crop == "WHEAT" else (8 if crop == "MELON" else 10)
        quantity = min(quantity, cap)
        reserve = 120.0 if day < 7 else 300.0
        affordable = min(quantity, int(max(0.0, money - reserve) // CROPS[crop]["seed"]))
        if affordable > 0:
            orders.append(["BUY_SEED", crop, affordable])
            money -= affordable * CROPS[crop]["seed"]

    return orders[:10]


def agent(obs, configuration=None):
    farms = _get(obs, "farms", []) or []
    player = int(_get(obs, "player", 0) or 0)
    if player < 0 or player >= len(farms):
        return {"farmer": ["PASS"], "hands": [], "market": []}
    farm = farms[player]
    private = _dict(_get(obs, "private", {}))
    farmer, hands = _unit_actions(obs, farm, private)
    return {"farmer": farmer, "hands": hands, "market": _market_orders(obs, farm, private)}


## Static publication checks

These checks make the notebook's live-policy structure auditable and reject common replay artifacts.

In [ ]:
source = Path("main.py").read_text(encoding="utf-8")
compile(source, "main.py", "exec")
tree = ast.parse(source)

long_ids = re.findall(r"\b\d{8,}\b", source)
largest_literal = max(
    (
        len(node.elts)
        for node in ast.walk(tree)
        if isinstance(node, (ast.List, ast.Tuple, ast.Set))
    ),
    default=0,
)

assert "def agent(obs, configuration=None):" in source
assert "EpisodeId" not in source
assert "submissionId" not in source
assert "TRACE_ACTIONS" not in source
assert not long_ids
assert largest_literal <= 20

print("Python syntax: PASS")
print("Agent entry point: PASS")
print("Replay trace scan: PASS")
print("Long numeric identifier scan: PASS")
print("Largest literal collection:", largest_literal)

## Official-environment smoke tests

The following tests run the agent from both seats and perform validation-style self-play. They are execution checks, not a guaranteed leaderboard rating.

In [ ]:
from kaggle_environments import make
import main
importlib.reload(main)

def passive_agent(obs, configuration=None):
    hand_count = len(obs["farms"][obs["player"]].get("hands", []))
    return {
        "farmer": ["PASS"],
        "hands": [["PASS"] for _ in range(hand_count)],
        "market": [],
    }

tests = [
    ("V3 / passive", [main.agent, passive_agent]),
    ("passive / V3", [passive_agent, main.agent]),
    ("V3 / V3", [main.agent, main.agent]),
]

for label, agents in tests:
    env = make("kaggriculture", configuration={"seed": 0}, debug=True)
    env.run(agents)
    final_step = env.steps[-1]
    rewards = [state.get("reward") for state in final_step]
    statuses = [state.get("status") for state in final_step]
    print(f"{label:15s} rewards={rewards} statuses={statuses}")

## Build the submission archive

The archive contains exactly one root-level file: `main.py`.

In [ ]:
archive = Path("submission.tar.gz")
with tarfile.open(archive, "w:gz") as tar:
    tar.add("main.py", arcname="main.py")

with tarfile.open(archive, "r:gz") as tar:
    members = tar.getnames()

assert members == ["main.py"]
print("Created:", archive.resolve())
print("Members:", members)
print("Archive size:", archive.stat().st_size, "bytes")

## Publication checklist

1. Create a new blank Kaggle notebook; do not use **Copy & Edit** on another participant's notebook.
2. Import this `.ipynb`.
3. Run **Save Version → Save & Run All**.
4. Confirm that every smoke-test status is `DONE`.
5. Keep the authorship, provenance, and licensing section.
6. Do not insert another participant's code, prose, screenshots, action traces, or identifiers.
7. Verify the competition's current Code Sharing and Rules pages before making the notebook public.
8. Report only results actually produced by Kaggle or your clearly labelled local tests.

No workflow can guarantee that content will never be reported, but this notebook is structured to make its independent origin and live decision process clear and auditable.